# IMTalker — Emotion LoRA Training (notebook 5/5)

Adds **emotion conditioning** to a pretrained IMTalker generator using targeted LoRA
fine-tuning. After training you can make the avatar express **neutral, happy, sad,
angry, surprise, or fear** by setting a single integer at inference time — no
architecture surgery on the real-time pipeline required.

**What this notebook does**
- Validates the environment (GPU, CUDA, dependencies, checkpoints, dataset)
- Launches emotion LoRA training as a background process (Option A, multi-GPU)
  or directly in the notebook (Option B, single GPU)
- Monitors training progress (loss log, checkpoints, test videos)
- Verifies the trained checkpoint is loadable and produces correct output

**Prerequisites**
1. Notebooks `01`–`03` completed for your emotion video dataset
   → `emotion_dataset/{motion,audio,smirk,gaze}/` each containing ~8000+ files
2. `preprocessing_output/checkpoints/renderer.ckpt` and `wav2vec2-base-960h/`
3. A pretrained base generator checkpoint (the one used in your live system)
4. The emotion conditioning code added to `generator/FM.py`, `generator/FMT.py`,
   `generator/train_lora.py` (added by the implementation step)
5. `generator/emotion_dataset.py` and `generator/run_emotion_lora_training.py` created

> **RunPod note**: this notebook is designed for Ubuntu pods with CUDA 12.x.
> It works on RTX 5090, A100, H100, and any other CUDA-capable GPU RunPod provides.

In [ ]:
!git clone https://github.com/MoshiHead/n_IMT_s_Perso_v2_2_aug_compress_log_Emotion_anti_v1.git speech2avatar

In [ ]:
# ============================================================================
# CONFIGURATION — EDIT THE PATHS AND SETTINGS BELOW
# ============================================================================
from pathlib import Path
import torch

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

# Repository root (the folder that contains generator/, renderer/, app.py, etc.)
REPO_ROOT = Path("/workspace/speech2avatar/IMTalker")

# Your emotion dataset root (flat layout: motion/ audio/ smirk/ gaze/)
# This is where you ran preprocessing notebooks 01-03 for your emotion videos
EMOTION_DATASET_DIR = Path("/workspace/emotion_dataset")

# Checkpoints directory (from preprocessing notebooks 01/02)
CKPT_DIR = REPO_ROOT.parent / "preprocessing_output" / "checkpoints"
WAV2VEC_DIR   = CKPT_DIR / "wav2vec2-base-960h"
RENDERER_CKPT = CKPT_DIR / "renderer.ckpt"

# Base generator checkpoint (the pretrained model used in your live system)
# This is the checkpoint the emotion LoRA is fine-tuned ON TOP OF.
PRETRAINED_GENERATOR_CKPT = Path(
    "/workspace/speech2avatar/checkpoints/live_winner/lora/"
    "ditto_blink_lora_withaudio_r64_096_continue_2h_last.ckpt"
)

# Where to store training outputs (logs, checkpoints, test videos)
EXP_ROOT = REPO_ROOT.parent / "emotion_exps"
EXP_NAME = "emotion_lora_v1"

# ---------------------------------------------------------------------------
# LoRA hyper-parameters
# ---------------------------------------------------------------------------
LORA_RANK    = 32      # LoRA rank for adaLN/decoder adapters
LORA_ALPHA   = 64.0   # LoRA alpha (scaling factor = alpha/rank)
LORA_DROPOUT = 0.05   # Dropout on LoRA paths during training

# ---------------------------------------------------------------------------
# Training hyper-parameters
# ---------------------------------------------------------------------------
BATCH_SIZE   = 8        # Per-GPU batch size (use 4-8 for 24GB; 16 for 80GB A100)
LR           = 1e-4     # Learning rate for LoRA params; emotion_conditioner uses 2x LR
MAX_STEPS    = 100_000  # Total training steps
MAX_EPOCHS   = -1       # -1 = stop at MAX_STEPS; set e.g. 50 to stop by epoch count
SAVE_FREQ    = 5_000    # Save checkpoint every N steps
DISPLAY_FREQ = 2_000    # Validation every N steps
LOG_EVERY    = 50       # Text log write interval in steps
NUM_WORKERS  = 4        # DataLoader worker processes

# How many samples to hold out for validation (from the same flat dataset)
VAL_SPLIT    = 200      # Increase if you have more data; must be < total samples

# ---------------------------------------------------------------------------
# Resume / restart
# ---------------------------------------------------------------------------
AUTO_RESUME  = True     # Resume from last.ckpt if present
RESUME_CKPT  = None     # Or: "/path/to/specific.ckpt"

# ---------------------------------------------------------------------------
# Emotion filter (optional)
# Set to None to train on ALL emotions. Or restrict e.g. ["happy", "sad"]
# Valid values: "neutral", "happy", "sad", "angry", "surprise", "fear"
# ---------------------------------------------------------------------------
EMOTION_FILTER = None   # e.g. ["happy", "sad", "neutral"]

# ---------------------------------------------------------------------------
# Per-epoch test video (optional but recommended)
# ---------------------------------------------------------------------------
TEST_REF_IMAGE    = None    # None = auto-pick from assets/
TEST_AUDIO        = None    # None = auto-pick from assets/
TEST_EVERY_N      = 1       # Render a test video every N epochs
TEST_CROP         = True    # Auto-crop face in the test image
TEST_CFG_SCALE    = 3.0
TEST_NFE          = 10
TEST_SEED         = 42

# ---------------------------------------------------------------------------
# Quick sanity prints
# ---------------------------------------------------------------------------
EXP_DIR = EXP_ROOT / EXP_NAME
print(f"repo:              {REPO_ROOT}")
print(f"emotion dataset:   {EMOTION_DATASET_DIR}")
print(f"pretrained ckpt:   {PRETRAINED_GENERATOR_CKPT}")
print(f"exp output:        {EXP_DIR}")
print(f"LoRA rank={LORA_RANK} alpha={LORA_ALPHA} | "
      f"batch={BATCH_SIZE} lr={LR} max_steps={MAX_STEPS:,}")

In [ ]:
# Check and install required packages (safe to re-run)
import importlib.util, shutil, subprocess, sys

def _pip(*pkgs):
    print("[install]", " ".join(pkgs))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs), check=True)

required = {
    "pytorch_lightning": "pytorch-lightning==2.2.1",
    "torchdiffeq":       "torchdiffeq==0.2.5",
    "einops":            "einops",
    "transformers":      "transformers>=4.38,<4.42",
    "librosa":           "librosa",
    "face_alignment":    "face_alignment==1.4.1",
    "av":                "av==12.0.0",
    "tensorboard":       "tensorboard",
    "timm":              "timm==1.0.9",
    "tqdm":              "tqdm",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    _pip(*missing)
else:
    print("All required packages already installed")

import torch
import pytorch_lightning as pl
print(f"torch:             {torch.__version__}")
print(f"CUDA available:    {torch.cuda.is_available()}")
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU[{i}] {props.name}  {props.total_memory / 1e9:.1f} GB")
print(f"pytorch-lightning: {pl.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found — emotion LoRA training requires a GPU.")
assert shutil.which("ffmpeg"), "ffmpeg not found on PATH (needed for test videos)"

In [ ]:
# Pre-flight checks: dataset, checkpoints, emotion code, test files
import cv2, sys

# 1) Make sure the repo is importable
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "generator"))

# 2) Check emotion conditioning code was added to the repo
try:
    from generator.FM import FMGenerator, EmotionConditioner
    print("\u2713 EmotionConditioner found in generator/FM.py")
except ImportError as e:
    raise ImportError(
        f"EmotionConditioner not found in generator/FM.py: {e}\n"
        "Make sure you ran the implementation step to add emotion conditioning."
    )

try:
    from generator.emotion_dataset import EmotionAudioMotionDataset, EMOTION_MAP, parse_emotion_from_stem
    print("\u2713 generator/emotion_dataset.py found")
except ImportError as e:
    raise ImportError(
        f"generator/emotion_dataset.py not found: {e}\n"
        "Make sure the emotion_dataset.py file was created in generator/"
    )

training_script = REPO_ROOT / "generator" / "run_emotion_lora_training.py"
assert training_script.exists(), (
    f"Missing {training_script}\n"
    "Make sure run_emotion_lora_training.py was created in generator/"
)
print(f"\u2713 {training_script.name} found")

from generator.train_lora import apply_emotion_lora_to_model
print("\u2713 apply_emotion_lora_to_model found in generator/train_lora.py")

# 3) Dataset
for sub in ("motion", "audio", "smirk", "gaze"):
    d = EMOTION_DATASET_DIR / sub
    assert d.exists(), f"Missing {d} — run preprocessing notebooks 01-03 for your emotion dataset"
counts = {s: len(list((EMOTION_DATASET_DIR / s).glob("*.*")))
          for s in ("motion", "audio", "smirk", "gaze")}
print(f"\nDataset file counts: {counts}")
assert counts["motion"] >= max(10, VAL_SPLIT + 1), \
    f"Need at least {VAL_SPLIT + 10} clips, found {counts['motion']}"

# 4) Emotion label distribution check
motion_files = sorted((EMOTION_DATASET_DIR / "motion").glob("*.pt"))
from collections import Counter
emotion_distribution = Counter()
for mp in motion_files:
    eid = parse_emotion_from_stem(mp.stem)
    emotion_distribution[eid] += 1
from generator.emotion_dataset import EMOTION_NAMES
print("\nEmotion distribution in dataset:")
for eid in range(6):
    cnt = emotion_distribution.get(eid, 0)
    bar = "\u2588" * (cnt // 50) if cnt > 0 else "(missing)"
    print(f"  [{eid}] {EMOTION_NAMES[eid]:10s}: {cnt:5d}  {bar}")
assert sum(emotion_distribution.values()) > 0, "No valid emotion labels found in dataset"

# 5) Checkpoints
assert WAV2VEC_DIR.exists() and any(WAV2VEC_DIR.iterdir()), \
    f"wav2vec2 not found at {WAV2VEC_DIR} — run notebook 02"
if not RENDERER_CKPT.exists():
    from huggingface_hub import hf_hub_download
    print("[download] renderer.ckpt from cbsjtu01/IMTalker...")
    hf_hub_download(repo_id="cbsjtu01/IMTalker", filename="renderer.ckpt",
                    local_dir=str(CKPT_DIR))
assert RENDERER_CKPT.exists(), f"renderer.ckpt not found at {RENDERER_CKPT}"
assert PRETRAINED_GENERATOR_CKPT.exists(), \
    f"Pretrained generator checkpoint not found: {PRETRAINED_GENERATOR_CKPT}\n" \
    "Set PRETRAINED_GENERATOR_CKPT in the CONFIG cell."
print(f"\n\u2713 wav2vec2: {WAV2VEC_DIR}")
print(f"\u2713 renderer: {RENDERER_CKPT}")
print(f"\u2713 pretrained generator: {PRETRAINED_GENERATOR_CKPT}")

# 6) Test reference image / audio (auto-pick)
if TEST_REF_IMAGE is None:
    cand = REPO_ROOT / "assets" / "source_3.jpg"
    if cand.exists():
        TEST_REF_IMAGE = cand
    else:
        vids = sorted((REPO_ROOT.parent / "preprocessing_output" / "videos_512").glob("*.mp4"))
        if vids:
            cap = cv2.VideoCapture(str(vids[0]))
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) // 2)
            ok, frm = cap.read(); cap.release()
            if ok:
                ref_out = EXP_DIR / "test_ref.png"
                ref_out.parent.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(ref_out), frm)
                TEST_REF_IMAGE = ref_out
if TEST_AUDIO is None:
    cand = REPO_ROOT / "assets" / "audio_1.wav"
    if cand.exists():
        TEST_AUDIO = cand

strategy = "ddp_find_unused_parameters_true" if n_gpu > 1 else "auto"
eff_batch = BATCH_SIZE * n_gpu
print(f"\n\u2713 gpus={n_gpu}  strategy={strategy}  effective batch={eff_batch}")
if TEST_REF_IMAGE:
    print(f"\u2713 test image: {TEST_REF_IMAGE}")
if TEST_AUDIO:
    print(f"\u2713 test audio: {TEST_AUDIO}")
print("\nAll pre-flight checks passed \u2713")

## Launch Training

**Option A (recommended — required for multi-GPU)**: Launches training as a
background subprocess with the repository's exact DDP configuration.
Survives notebook/kernel restarts. Writes `train_console.log` + `train_log.txt`.

**Option B (single GPU only)**: Runs the training directly in this notebook cell.
Interactive progress bar; the cell blocks until training finishes or is interrupted.

Both options use the same `generator/run_emotion_lora_training.py` launcher.

> **Tip**: With ~8,000 samples and `batch_size=8` on a single RTX 5090 (32 GB),
> 100,000 steps takes roughly **2–4 hours** depending on NFE and num_workers.

In [ ]:
# OPTION A — Launch emotion LoRA training as a background process.
# Required for multi-GPU; survives kernel restarts.
import os, subprocess, sys

training_script = REPO_ROOT / "generator" / "run_emotion_lora_training.py"
console_log = EXP_DIR / "train_console.log"
EXP_DIR.mkdir(parents=True, exist_ok=True)

cmd = [sys.executable, str(training_script),
    "--repo_root",          str(REPO_ROOT),
    "--dataset_path",       str(EMOTION_DATASET_DIR),
    "--pretrained_ckpt",    str(PRETRAINED_GENERATOR_CKPT),
    "--exp_root",           str(EXP_ROOT),
    "--exp_name",           EXP_NAME,
    "--wav2vec_model_path", str(WAV2VEC_DIR),
    "--renderer_ckpt",      str(RENDERER_CKPT),
    "--lora_rank",          str(LORA_RANK),
    "--lora_alpha",         str(LORA_ALPHA),
    "--lora_dropout",       str(LORA_DROPOUT),
    "--batch_size",         str(BATCH_SIZE),
    "--lr",                 str(LR),
    "--max_steps",          str(MAX_STEPS),
    "--max_epochs",         str(MAX_EPOCHS),
    "--save_freq",          str(SAVE_FREQ),
    "--display_freq",       str(DISPLAY_FREQ),
    "--log_every_n_steps",  str(LOG_EVERY),
    "--num_workers",        str(NUM_WORKERS),
    "--val_split",          str(VAL_SPLIT),
    "--test_every_n_epochs", str(TEST_EVERY_N),
    "--test_crop",          "1" if TEST_CROP else "0",
    "--test_cfg",           str(TEST_CFG_SCALE),
    "--test_nfe",           str(TEST_NFE),
    "--test_seed",          str(TEST_SEED),
]
if TEST_REF_IMAGE:
    cmd += ["--test_ref", str(TEST_REF_IMAGE)]
if TEST_AUDIO:
    cmd += ["--test_aud", str(TEST_AUDIO)]
if TEST_EVERY_N:
    cmd += ["--test_out_dir", str(EXP_DIR / "test_videos")]
if not AUTO_RESUME:
    cmd.append("--no_auto_resume")
if RESUME_CKPT:
    cmd += ["--resume_ckpt", str(RESUME_CKPT)]
if EMOTION_FILTER:
    cmd += ["--emotion_filter"] + list(EMOTION_FILTER)

print("Launching:", " ".join(cmd[:8]), "...\n")
with open(console_log, "a") as logf:
    kwargs = dict(stdout=logf, stderr=subprocess.STDOUT, cwd=str(REPO_ROOT))
    if os.name == "posix":
        kwargs["start_new_session"] = True
    else:
        kwargs["creationflags"] = subprocess.CREATE_NEW_PROCESS_GROUP
    proc = subprocess.Popen(cmd, **kwargs)

(EXP_DIR / "train.pid").write_text(str(proc.pid))
print(f"Training started  | pid={proc.pid}")
print(f"Console log:      {console_log}")
print(f"Checkpoint dir:   {EXP_DIR / 'checkpoints'}")
print("Run the MONITOR cell below to follow progress.")

In [ ]:
# OPTION B — Train directly in this notebook (single GPU only).
# This cell blocks until training finishes. For multi-GPU, use OPTION A.
assert torch.cuda.device_count() <= 1, \
    f"{torch.cuda.device_count()} GPUs detected — use OPTION A for multi-GPU training."

import sys
for p in (str(REPO_ROOT), str(REPO_ROOT / "generator")):
    if p not in sys.path:
        sys.path.insert(0, p)

# Add preprocessing scripts dir so imtalker_train_utils is importable
preprocessing_dir = REPO_ROOT / "preprocessing"
if preprocessing_dir.exists() and str(preprocessing_dir) not in sys.path:
    sys.path.insert(0, str(preprocessing_dir))

import argparse
import importlib.util, importlib

spec = importlib.util.spec_from_file_location(
    "run_emotion_lora_training",
    str(REPO_ROOT / "generator" / "run_emotion_lora_training.py")
)
relt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(relt)

args = argparse.Namespace(
    repo_root=str(REPO_ROOT),
    dataset_path=str(EMOTION_DATASET_DIR),
    pretrained_ckpt=str(PRETRAINED_GENERATOR_CKPT),
    exp_root=str(EXP_ROOT),
    exp_name=EXP_NAME,
    wav2vec_model_path=str(WAV2VEC_DIR),
    renderer_ckpt=str(RENDERER_CKPT),
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    batch_size=BATCH_SIZE,
    lr=LR,
    max_steps=MAX_STEPS,
    max_epochs=MAX_EPOCHS,
    save_freq=SAVE_FREQ,
    display_freq=DISPLAY_FREQ,
    log_every_n_steps=LOG_EVERY,
    num_workers=NUM_WORKERS,
    val_split=VAL_SPLIT,
    resume_ckpt=str(RESUME_CKPT) if RESUME_CKPT else None,
    no_auto_resume=not AUTO_RESUME,
    test_ref=str(TEST_REF_IMAGE) if TEST_REF_IMAGE else None,
    test_aud=str(TEST_AUDIO) if TEST_AUDIO else None,
    test_out_dir=str(EXP_DIR / "test_videos"),
    test_every_n_epochs=TEST_EVERY_N,
    test_crop=1 if TEST_CROP else 0,
    test_cfg=TEST_CFG_SCALE,
    test_nfe=TEST_NFE,
    test_seed=TEST_SEED,
    emotion_filter=EMOTION_FILTER,
    list_path=None,
    val_list_path=None,
)
relt.main(args)

In [ ]:
# MONITOR — re-run this cell anytime to check training progress
import os
from pathlib import Path

def _tail(path, n=20):
    path = Path(path)
    if not path.exists():
        print(f"  (no {path.name} yet)")
        return
    lines = path.read_text(errors="replace").splitlines()
    print(f"----- last {min(n, len(lines))} lines of {path.name} -----")
    print("\n".join(lines[-n:]))

# Process status (Option A)
pid_file = EXP_DIR / "train.pid"
if pid_file.exists():
    pid = int(pid_file.read_text().strip())
    try:
        import psutil
        alive = psutil.pid_exists(pid)
    except ImportError:
        try:
            os.kill(pid, 0); alive = True
        except OSError:
            alive = False
    status = "RUNNING \u2713" if alive else "finished / not running"
    print(f"Training process pid={pid}: {status}")
else:
    print("No train.pid file (use Option A to see process status)")

print()
_tail(EXP_DIR / "train_log.txt", 25)
print()

# Checkpoints
cks = sorted((EXP_DIR / "checkpoints").glob("*.ckpt"),
             key=lambda p: p.stat().st_mtime) if (EXP_DIR / "checkpoints").exists() else []
print(f"\nCheckpoints ({len(cks)}): {[c.name for c in cks[-5:]]}")

# Test videos
tvs = sorted((EXP_DIR / "test_videos").glob("epoch_*.mp4")) if (EXP_DIR / "test_videos").exists() else []
print(f"Test videos  ({len(tvs)}): {[t.name for t in tvs[-4:]]}")
if tvs:
    print(f"Latest: {tvs[-1]}")

# CSV metrics
csvs = sorted(EXP_DIR.glob("version_*/metrics.csv"))
if csvs:
    _tail(csvs[-1], 5)

In [ ]:
# STOP training started with OPTION A (sends SIGTERM; last checkpoint saved)
STOP_TRAINING = False  # set to True and re-run to actually stop

if not STOP_TRAINING:
    print("Set STOP_TRAINING = True and re-run this cell to stop training.")
else:
    import os, signal, subprocess
    pid_file = EXP_DIR / "train.pid"
    assert pid_file.exists(), "No train.pid file — was training launched with OPTION A?"
    pid = int(pid_file.read_text().strip())
    if os.name == "posix":
        os.kill(pid, signal.SIGTERM)
    else:
        subprocess.run(["taskkill", "/PID", str(pid), "/T", "/F"])
    print(f"Stop signal sent to pid {pid}")

## Validate the Trained Checkpoint

Run the cells below **after** training finishes to verify:
1. The checkpoint is loadable and contains emotion conditioning weights
2. The emotion_conditioner produces different biases for each emotion
3. The model generates distinct motion for each emotion using the same audio

In [ ]:
# Load the trained emotion LoRA checkpoint and verify its contents
import torch, sys
from pathlib import Path

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "generator"))

# Pick the latest checkpoint (or set explicitly)
EVAL_CKPT = None  # None = auto-pick last.ckpt

if EVAL_CKPT is None:
    last_ckpt = EXP_DIR / "checkpoints" / "last.ckpt"
    step_ckpts = sorted((EXP_DIR / "checkpoints").glob("step=*.ckpt"),
                        key=lambda p: p.stat().st_mtime)
    EVAL_CKPT = last_ckpt if last_ckpt.exists() else (step_ckpts[-1] if step_ckpts else None)

assert EVAL_CKPT and Path(EVAL_CKPT).exists(), \
    f"No checkpoint found in {EXP_DIR / 'checkpoints'}"
print(f"Evaluating: {EVAL_CKPT}")

ckpt = torch.load(str(EVAL_CKPT), map_location="cpu", weights_only=False)
state = ckpt.get("ema_state_dict") or ckpt.get("state_dict", ckpt)
if isinstance(state, dict) and "model" in state and isinstance(state["model"], dict):
    state = state["model"]
state = {k.replace("model.", "", 1) if k.startswith("model.") else k: v for k, v in state.items()}

# Check emotion-specific keys
emotion_keys = [k for k in state if "emotion" in k]
lora_keys    = [k for k in state if "lora_"  in k]
print(f"\nEmotion conditioner keys: {len(emotion_keys)}")
for k in emotion_keys:
    print(f"  {k}: {tuple(state[k].shape)}")
print(f"\nLoRA adapter keys: {len(lora_keys)}")
print(f"Total keys in checkpoint: {len(state)}")

# Check that emotion_projection weights are NON-ZERO (training happened)
proj_w = state.get("emotion_conditioner.emotion_projection.0.weight")
if proj_w is not None:
    norm = proj_w.norm().item()
    print(f"\nemotion_projection weight norm: {norm:.6f}")
    if norm < 1e-6:
        print("  WARNING: weights still zero — training may not have converged yet")
    else:
        print("  \u2713 Emotion projection learned (norm > 0)")

In [ ]:
# Verify the emotion conditioner produces distinct, non-zero biases per emotion
import torch, sys
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "generator"))

import argparse
from generator.options.base_options import BaseOptions
from generator.FM import FMGenerator, EmotionConditioner, EMOTION_MAP, EMOTION_NAMES
from generator.train_lora import apply_emotion_lora_to_model

# Build model with same opt as training
parser = argparse.ArgumentParser()
parser = BaseOptions().initialize(parser)
parser.add_argument("--skip_fm_audio_encoder", action="store_true")
opt = parser.parse_args([])
opt.rank = "cpu"
opt.wav2vec_model_path = str(WAV2VEC_DIR)
opt.skip_fm_audio_encoder = True

model = FMGenerator(opt)
apply_emotion_lora_to_model(model, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=0.0)
missing, unexpected = model.load_state_dict(state, strict=False)
model.eval()
print(f"Model loaded | missing={len(missing)} unexpected={len(unexpected)}")

# Check per-emotion embeddings
print("\nEmotion bias norms (T=50 frames):")
biases = {}
for name, idx in EMOTION_MAP.items():
    eid = torch.tensor([idx], dtype=torch.long)
    with torch.no_grad():
        bias = model.emotion_conditioner(eid, T=50, train=False)  # [1, 50, 32]
    norm = bias.mean(dim=1).norm().item()
    biases[name] = bias.squeeze(0).mean(0)  # [32]
    print(f"  [{idx}] {name:10s}: norm={norm:.6f}")

# Pairwise distances (happy vs sad should be far; happy vs happy = 0)
print("\nPairwise L2 distances between emotion embeddings (mean over 32 dims):")
names = list(EMOTION_MAP.keys())
for i, n1 in enumerate(names):
    for j, n2 in enumerate(names):
        if j <= i:
            continue
        dist = (biases[n1] - biases[n2]).norm().item()
        print(f"  {n1:10s} vs {n2:10s}: {dist:.4f}")

## After Training — Output Files

All outputs are in `{EXP_ROOT}/{EXP_NAME}/`:

| Path | Content |
|------|---------|
| `checkpoints/last.ckpt` | Latest checkpoint (auto-resume source) |
| `checkpoints/step=XXXXXX.ckpt` | Periodic checkpoints (every `SAVE_FREQ` steps) |
| `train_log.txt` | Human-readable epoch/step/loss/lr log |
| `train_console.log` | Full console output (Option A) |
| `version_*/metrics.csv` | TensorBoard + CSV metrics |
| `test_videos/epoch_XXXX.mp4` | Per-epoch test videos |

## Using the Trained Checkpoint in Your Live System

Pass these arguments to your live server (in addition to your existing args):

```bash
# In run_live.sh or start_winner_live.sh, set:
EMOTION_LORA_PATH="/path/to/emotion_exps/emotion_lora_v1/checkpoints/last.ckpt"
EMOTION_LORA_RANK=32
EMOTION_LORA_ALPHA=64.0
EMOTION_ID=1   # 0=neutral 1=happy 2=sad 3=angry 4=surprise 5=fear
```

The emotion is controlled by `--emotion_id` in the live server.
See the companion notebook `06_emotion_live_pipeline.ipynb` for the full
emotion-aware RunPod deployment.